# 处理时间计算 (Processing Time Computation)

本 notebook 用于计算不同窗口大小下的处理时间，包括：
- 单个 clique 的处理时间
- 所有 clique 串行处理的总时间


## 1. 导入必要的库


In [1]:
import numpy as np
import pandas as pd
import pickle
import warnings
import gc
import time
from pathlib import Path

warnings.filterwarnings('ignore')

import torch
import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend to save memory
import matplotlib.pyplot as plt
import seaborn as sns

# Try to import tqdm for progress bars
try:
    from tqdm import tqdm
    HAS_TQDM = True
except ImportError:
    HAS_TQDM = False
    # Simple tqdm replacement
    class tqdm:
        def __init__(self, iterable=None, total=None, desc=None, leave=True, unit=None):
            self.iterable = iterable
            self.total = total
            self.desc = desc or ""
            self.leave = leave
            self.unit = unit or "it"
            self.current = 0
            if iterable is not None:
                self.iterable = iterable
            else:
                self.iterable = range(total) if total else []
        
        def __enter__(self):
            if self.desc:
                print(f"{self.desc}: ", end="", flush=True)
            return self
        
        def __exit__(self, *args):
            print()
        
        def __iter__(self):
            for item in self.iterable:
                self.current += 1
                if self.total and self.current % max(1, self.total // 20) == 0:
                    print(f"{self.desc}: {self.current}/{self.total} ({100*self.current/self.total:.1f}%)", end="\r", flush=True)
                yield item
        
        def update(self, n=1):
            self.current += n
            if self.total and self.current % max(1, self.total // 20) == 0:
                print(f"{self.desc}: {self.current}/{self.total} ({100*self.current/self.total:.1f}%)", end="\r", flush=True)
        
        def close(self):
            if self.total:
                print(f"{self.desc}: {self.current}/{self.total} (100.0%)")

from utils_clique import (
    build_sliding_cliques,
    SimpleAutoSort,
    SimpleWaveformLoader,
    detect_spike
)


## 2. 配置参数


In [2]:
# 数据路径配置
recording_path = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s.h5"
base_save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/"

# 测试参数
window_sizes_ms = [100, 200, 300, 400, 500, 1000]  # 毫秒
n_runs_per_window = 20  # 每个窗口大小的运行次数

# 检测参数
detection_params = {
    'thr_min': 2.7,
    'thr_max': 15,
    'distance': 4,
    'ch_max_simul_firing': 8,
    'wlen': 6,
    'prominence': 10,
}

# 窗口参数
window_params = {
    'left_sample': 10,
    'right_sample': 20,
}

# 评估时间段（用于测试）
eval_time_segments = [
    (600, 1200),   # Segment 0: 600-1200 seconds
    (1200, 1800),  # Segment 1: 1200-1800 seconds
    (1800, 2400),  # Segment 2: 1800-2400 seconds
    (2400, 3000),  # Segment 3: 2400-3000 seconds
    (3000, 3600),  # Segment 4: 3000-3600 seconds
]

# 使用第一个时间段进行测试
test_segment_id = 0

# 使用的模型运行ID
run_id = 1

# 设备配置
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


Using device: cuda


## 3. 加载数据


In [3]:
print("=" * 80)
print("加载数据")
print("=" * 80)

# 加载 recording
print("Loading recording...")
recording, sorting = se.read_mearec(recording_path)

# 获取 probe
probe = recording.get_probe()
if probe is None:
    raise ValueError("Recording does not have probe information")

# 加载 clique 信息
clique_info_path = Path(base_save_dir) / "clique_info.pkl"

if clique_info_path.exists():
    with open(clique_info_path, 'rb') as f:
        clique_info = pickle.load(f)
    cliques = clique_info['cliques']
    print(f"Loaded {len(cliques)} cliques from {clique_info_path}")
else:
    # 从 probe 构建 cliques
    cliques = build_sliding_cliques(
        probe,
        clique_size=49,
        min_size=25,
        min_overlap=18,
        target_groups=12,
    )
    print(f"Built {len(cliques)} cliques")

# 预处理 recording
recording_f = recording.rename_channels(range(384))

print(f"\nRecording loaded successfully")
print(f"Sampling rate: {recording_f.get_sampling_frequency()} Hz")
print(f"Number of channels: {recording_f.get_num_channels()}")
print(f"Recording duration: {recording_f.get_num_samples() / recording_f.get_sampling_frequency():.2f} seconds")


加载数据
Loading recording...
Loaded 12 cliques from /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_info.pkl

Recording loaded successfully
Sampling rate: 10000.0 Hz
Number of channels: 384
Recording duration: 3600.00 seconds


## 4. 准备测试时间段


In [4]:
start_time, end_time = eval_time_segments[test_segment_id]
duration_seconds = end_time - start_time
sampling_rate = recording_f.get_sampling_frequency()
start_sample = int(start_time * sampling_rate)
end_sample = int(end_time * sampling_rate)
recording_segment = recording_f.frame_slice(start_frame=start_sample, end_frame=end_sample)

print(f"Test segment: {test_segment_id} ({start_time}-{end_time} seconds)")
print(f"Processing window sizes to test: {window_sizes_ms} ms")
print(f"Number of runs per window size: {n_runs_per_window}")


Test segment: 0 (600-1200 seconds)
Processing window sizes to test: [100, 200, 300, 400, 500, 1000] ms
Number of runs per window size: 20


## 5. 加载模型和校准结果


In [7]:
print("=" * 80)
print("加载模型和校准结果")
print("=" * 80)
print(f"Loading models and calibration results for run {run_id}...")

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

clique_models = {}
clique_calibration_results = {}

for clique_id in tqdm(range(len(cliques)), desc="Loading models"):
    model_save_dir = Path(base_save_dir) / f"clique_{clique_id:02d}" / "model_save" / f"run_{run_id}"
    
    if not model_save_dir.exists():
        continue
    
    try:
        # 加载模型
        n_channels = len(cliques[clique_id].device_channel_indices)
        train_data_dir = model_save_dir.parent.parent.parent / f"clique_{clique_id:02d}" / "train_data"
        
        if not train_data_dir.exists():
            continue
            
        dataset = SimpleWaveformLoader(
            root=str(train_data_dir) + "/",
            shank_channel=np.arange(n_channels),
            Keep_id=None
        )
        
        # 获取单元数（从 keep_id 列表的长度）
        n_units = len(dataset.keep_id)
        set_shank_id = dataset.keep_id  # 单元 ID 列表，用于确定分类器输出维度
        
        autosort_model = SimpleAutoSort(
            ch_num=n_channels,
            samplepoints=window_params['left_sample'] + window_params['right_sample'],
            device=device,
            set_shank_id=set_shank_id,
            save_dir=str(model_save_dir) + "/",
            pos_weight_noise=dataset.pos_weight_noise.to(device),
            pos_weight_label=dataset.pos_weight_label.to(device)
        )
        autosort_model.load_model()
        autosort_model.eval()
        clique_models[clique_id] = autosort_model
        
        # 加载校准结果（如果可用）
        calibration_path = model_save_dir / "calibration_results.pkl"
        if calibration_path.exists():
            with open(calibration_path, 'rb') as f:
                clique_calibration_results[clique_id] = pickle.load(f)
        else:
            # 如果没有校准结果，创建一个模拟的校准结果
            print(f"  Clique {clique_id}: No calibration results found, creating mock calibration...")
            
            # 创建模拟的 PCA 模型（不进行降维，只是单位变换）
            # way3_features 通常是 100 维，PCA 降到 30 维
            mock_pca = PCA(n_components=30, random_state=42)
            # 使用随机数据拟合 PCA（仅用于初始化）
            mock_features = np.random.randn(1000, 100).astype(np.float32)
            mock_pca.fit(mock_features)
            
            # 创建模拟的 KMeans 模型
            # 使用实际的单元数
            n_units = len(dataset.keep_id)
            n_clusters = max(n_units, 10)  # 至少 10 个聚类
            mock_kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
            # 使用随机数据拟合 KMeans（仅用于初始化）
            mock_pca_features = np.random.randn(1000, 30).astype(np.float32)
            mock_kmeans.fit(mock_pca_features)
            
            # 创建模拟的映射（cluster_id -> unit_id）
            cluster_to_neuron_mapping = {}
            for i in range(n_clusters):
                # 随机映射到某个 unit_id，或者映射到 -1（噪声）
                if i < n_units:
                    cluster_to_neuron_mapping[i] = dataset.keep_id[i] if i < len(dataset.keep_id) else -1
                else:
                    cluster_to_neuron_mapping[i] = -1
            
            clique_calibration_results[clique_id] = {
                'kmeans_model': mock_kmeans,
                'pca_model': mock_pca,
                'cluster_to_neuron_mapping': cluster_to_neuron_mapping
            }
    
    except Exception as e:
        print(f"  Error loading model for clique {clique_id}: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\nLoaded {len(clique_models)} models and {len(clique_calibration_results)} calibration results")

# 选择同时有模型和校准结果的 cliques
available_cliques = [cid for cid in clique_models.keys() if cid in clique_calibration_results]
print(f"Available cliques for testing: {available_cliques}")

if len(available_cliques) == 0:
    raise ValueError("No cliques with both model and calibration results available!")


加载模型和校准结果
Loading models and calibration results for run 1...


Loading models:   0%|          | 0/12 [00:00<?, ?it/s]

Dataset loaded:
  - Total samples: 418358
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 9
  - Noise samples: 401878.0
  - Non-noise samples: 16480.0


Loading models:   8%|▊         | 1/12 [00:01<00:19,  1.73s/it]

  Clique 0: No calibration results found, creating mock calibration...
Dataset loaded:
  - Total samples: 415585
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 12
  - Noise samples: 398218.0
  - Non-noise samples: 17367.0
  Clique 1: No calibration results found, creating mock calibration...


...

Dataset loaded:
  - Total samples: 424007
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 12
  - Noise samples: 411516.0
  - Non-noise samples: 12491.0
  Clique 10: No calibration results found, creating mock calibration...


Loading models:  92%|█████████▏| 11/12 [02:13<00:14, 14.33s/it]

Dataset loaded:
  - Total samples: 417061
  - Number of channels: 49
  - Window length: 30
  - Number of unique units: 9
  - Noise samples: 407630.0
  - Non-noise samples: 9431.0
  Clique 11: No calibration results found, creating mock calibration...


Loading models: 100%|██████████| 12/12 [02:27<00:00, 12.31s/it]


Loaded 12 models and 12 calibration results
Available cliques for testing: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]


In [21]:
def process_single_clique_timing(
    recording_clique,
    autosort_model,
    calibration_results,
    start_frame,
    time_window_seconds,
    detection_params,
    window_params,
    device=None,
    return_n_spikes=False,  # 是否返回检测到的 spikes 数量
):
    """处理单个 clique 的时间窗口并测量处理时间"""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    left_sample = window_params['left_sample']
    right_sample = window_params['right_sample']
    window_size = left_sample + right_sample
    sampling_frequency = recording_clique.get_sampling_frequency()
    
    # 从校准阶段获取模型和映射
    kmeans_model = calibration_results['kmeans_model']
    pca_model = calibration_results['pca_model']
    cluster_to_neuron_mapping = calibration_results['cluster_to_neuron_mapping']
    
    # 开始计时：从阈值检测开始
    start_time = time.time()
    
    # 1. 加载当前窗口数据
    # start_frame 应该是相对于 recording_clique 的索引（因为 recording_clique 是从 recording_segment 选择的通道）
    window_frames = int(time_window_seconds * sampling_frequency)
    max_samples = recording_clique.get_num_samples()
    end_frame = min(start_frame + window_frames, max_samples)
    
    # 确保 end_frame > start_frame 且 start_frame 在有效范围内
    if start_frame < 0:
        start_frame = 0
    if start_frame >= max_samples:
        start_frame = max_samples - 1
    if end_frame <= start_frame:
        end_frame = min(start_frame + 1, max_samples)
    
    # 再次检查
    if end_frame <= start_frame:
        end_time = time.time()
        if return_n_spikes:
            return end_time - start_time, {
                'n_spikes_detected': 0,
                'n_waveforms_extracted': 0,
                'n_spikes_after_noise_classifier': 0,
                'n_timepoints': 0,
                'n_channels': recording_clique.get_num_channels(),
            }
        return end_time - start_time
    
    # get_traces 返回 (n_samples, n_channels)，不需要转置
    traces = recording_clique.get_traces(start_frame=start_frame, end_frame=end_frame).astype(np.float32)
    # traces 现在是 (n_samples, n_channels) = (n_timepoints, n_channels)
    
    # 2. 阈值检测
    # trace0_car 应该是 (n_timepoints, n_channels)，traces 已经是这个格式了
    trace0_car = traces  # (n_timepoints, n_channels)
    
    # 验证数据形状
    n_clique_channels = recording_clique.get_num_channels()
    if trace0_car.shape[1] != n_clique_channels:
        raise ValueError(f"通道数不匹配: trace0_car.shape[1]={trace0_car.shape[1]}, "
                        f"recording_clique.get_num_channels()={n_clique_channels}")
    
    # 阈值检测
    spikes = detect_spike(trace0_car, **detection_params)
    
    # 检查 spikes 的形状
    if spikes.shape != trace0_car.shape:
        raise ValueError(f"spikes 形状不匹配: spikes.shape={spikes.shape}, "
                        f"trace0_car.shape={trace0_car.shape}")
    
    spike_coords = np.argwhere(spikes == 1)  # (n_spikes, 2) [time, channel]
    
    # 检查是否有无效的通道索引
    if len(spike_coords) > 0:
        invalid_channels = spike_coords[:, 1][(spike_coords[:, 1] < 0) | (spike_coords[:, 1] >= n_clique_channels)]
        if len(invalid_channels) > 0:
            # 只记录第一次出现的错误，避免输出过多
            pass  # 会在下面的循环中跳过
    
    # 调试信息：显示检测结果
    n_spikes_detected = len(spike_coords)
    n_timepoints = trace0_car.shape[0]
    n_channels = trace0_car.shape[1]
    
    # 即使没有检测到 spikes，也继续计时（因为数据加载和检测本身也需要时间）
    # if len(spike_coords) == 0:
    #     return 0.0
    
    # 3. 提取波形并过滤边界
    waveforms = []
    spike_times = []
    spike_channels = []
    
    # 确保通道索引在有效范围内
    max_channel_idx = trace0_car.shape[1] - 1
    
    for time_idx, channel_idx in spike_coords:
        # 检查通道索引是否在有效范围内
        if channel_idx < 0 or channel_idx > max_channel_idx:
            continue  # 跳过无效的通道索引
        
        global_time_idx = start_frame + time_idx
        local_start = time_idx - left_sample
        local_end = time_idx + right_sample
        
        if local_start < 0 or local_end > trace0_car.shape[0]:
            continue
        if local_end - local_start != window_size:
            continue
        
        # traces 的形状是 (n_timepoints, n_channels)
        # waveform 应该是 (n_channels, window_size)
        # 切片 traces[local_start:local_end, :] 得到 (window_size, n_channels)，然后转置得到 (n_channels, window_size)
        waveform = traces[local_start:local_end, :].T  # 转置得到 (n_channels, window_size)
        waveforms.append(waveform)
        spike_times.append(global_time_idx)
        spike_channels.append(channel_idx)
    
    # 如果没有提取到有效波形，仍然继续计时（但跳过后续处理）
    n_waveforms_extracted = len(waveforms)
    
    if len(waveforms) == 0:
        end_time = time.time()
        if return_n_spikes:
            return end_time - start_time, {
                'n_spikes_detected': n_spikes_detected,
                'n_waveforms_extracted': n_waveforms_extracted,
                'n_spikes_after_noise_classifier': 0,
                'n_timepoints': n_timepoints,
                'n_channels': n_channels,
            }
        return end_time - start_time
    
    waveforms = np.array(waveforms)  # (n_spikes, n_channels, window_size)
    
    # 4. 通过噪声分类器，分类为 spike
    batch_size = 512
    n_spikes = len(waveforms)
    way3_features_list = []
    
    autosort_model.eval()
    with torch.no_grad():
        for i in range(0, n_spikes, batch_size):
            batch_end = min(i + batch_size, n_spikes)
            batch_waveforms = waveforms[i:batch_end]
            batch_channels = spike_channels[i:batch_end]
            
            batch_single_waveforms = []
            batch_multi_waveforms = []
            
            for wf, ch in zip(batch_waveforms, batch_channels):
                multi_wf = wf.flatten()
                batch_multi_waveforms.append(multi_wf)
                single_wf = wf[ch, :]
                batch_single_waveforms.append(single_wf)
            
            batch_multi_waveforms = np.array(batch_multi_waveforms)
            batch_single_waveforms = np.array(batch_single_waveforms)
            
            batch_multi = torch.from_numpy(batch_multi_waveforms).float().to(device)
            batch_single = torch.from_numpy(batch_single_waveforms).float().to(device)
            
            codes = torch.cat((batch_multi, batch_single), dim=1)
            
            noise_output = autosort_model.clsfier_noise(codes)
            noise_pred = torch.argmax(noise_output, dim=1)
            
            spike_mask = noise_pred == 1
            if spike_mask.sum() > 0:
                codes_spike = codes[spike_mask]
                way3_batch = autosort_model.clsfier_label.intermediate_forward(codes_spike)
                way3_features_list.append(way3_batch.cpu().numpy())
    
    if len(way3_features_list) == 0:
        end_time = time.time()
        return end_time - start_time
    
    way3_features = np.concatenate(way3_features_list, axis=0)
    n_spikes_after_noise_classifier = len(way3_features)
    
    # 5. PCA 降维
    way3_pca = pca_model.transform(way3_features)
    
    # 6. K-means 预测
    cluster_labels = kmeans_model.predict(way3_pca)
    
    # 7. 映射到训练神经元 ID（我们不需要实际预测用于计时）
    
    end_time = time.time()
    if return_n_spikes:
        return end_time - start_time, {
            'n_spikes_detected': n_spikes_detected,
            'n_waveforms_extracted': n_waveforms_extracted,
            'n_spikes_after_noise_classifier': n_spikes_after_noise_classifier,
            'n_timepoints': n_timepoints,
            'n_channels': n_channels,
        }
    return end_time - start_time


## 6.5. 测试阈值检测（调试用）


In [22]:
# 测试阈值检测是否工作
print("=" * 80)
print("测试阈值检测")
print("=" * 80)

# 选择一个 clique 进行测试
test_clique_id = available_cliques[0] if len(available_cliques) > 0 else 0
test_clique = cliques[test_clique_id]
clique_channels = list(set(test_clique.device_channel_indices))
recording_test = recording_segment.select_channels(channel_ids=clique_channels)

print(f"测试 Clique {test_clique_id}")
print(f"通道数: {len(clique_channels)}")
print(f"检测参数: {detection_params}")

# 测试几个随机的时间窗口
n_test_windows = 5
# 获取 recording_test 的总样本数（相对于 recording_segment）
recording_test_num_samples = recording_test.get_num_samples()
print(f"Recording test 总样本数: {recording_test_num_samples}")
print(f"Recording segment 总样本数: {recording_segment.get_num_samples()}")
print(f"时间范围: {start_time} - {end_time} 秒")
print(f"采样率: {sampling_rate} Hz\n")

for i in range(n_test_windows):
    # 随机选择一个时间窗口（100ms）
    # 使用相对于 recording_segment 的时间
    random_start_time = np.random.uniform(start_time, end_time - 0.1)
    # 计算相对于 recording_segment 的帧索引
    start_frame_absolute = int(random_start_time * sampling_rate)
    start_frame_relative = start_frame_absolute - start_sample  # 转换为相对于 recording_segment 的索引
    
    # 确保索引在有效范围内
    if start_frame_relative < 0:
        start_frame_relative = 0
    if start_frame_relative >= recording_test_num_samples:
        start_frame_relative = recording_test_num_samples - 1
    
    window_frames = int(0.1 * sampling_rate)  # 100ms
    end_frame_relative = min(start_frame_relative + window_frames, recording_test_num_samples)
    
    # 确保 end_frame > start_frame
    if end_frame_relative <= start_frame_relative:
        end_frame_relative = min(start_frame_relative + 1, recording_test_num_samples)
    
    start_frame_test = start_frame_relative
    end_frame_test = end_frame_relative
    
    # 加载数据
    # get_traces 返回 (n_samples, n_channels)，不需要转置
    traces = recording_test.get_traces(start_frame=start_frame_test, end_frame=end_frame_test).astype(np.float32)
    print(f"  traces 形状 (n_samples, n_channels): {traces.shape}")
    
    # 阈值检测
    # trace0_car 应该是 (n_timepoints, n_channels)
    trace0_car = traces  # (n_timepoints, n_channels)
    spikes = detect_spike(trace0_car, **detection_params)
    spike_coords = np.argwhere(spikes == 1)
    
    print(f"\n测试窗口 {i+1}:")
    print(f"  时间范围: {random_start_time:.2f} - {random_start_time + 0.1:.2f} 秒")
    print(f"  start_frame: {start_frame_test}, end_frame: {end_frame_test}")
    print(f"  窗口帧数: {end_frame_test - start_frame_test}")
    print(f"  traces 形状 (n_samples, n_channels): {traces.shape}")
    print(f"  时间点数: {trace0_car.shape[0]}, 通道数: {trace0_car.shape[1]}")
    print(f"  检测到的 spikes: {len(spike_coords)}")
    print(f"  Spikes 矩阵形状: {spikes.shape}, 1 的数量: {np.sum(spikes)}")
    
    if len(spike_coords) > 0:
        print(f"  前5个 spike 位置: {spike_coords[:5]}")
        print(f"  Spikes 矩阵统计: min={spikes.min()}, max={spikes.max()}, sum={spikes.sum()}")

print("\n" + "=" * 80)
print("如果所有窗口都检测到 0 个 spikes，可能需要调整检测参数")
print("=" * 80)


测试阈值检测
测试 Clique 0
通道数: 49
检测参数: {'thr_min': 2.7, 'thr_max': 15, 'distance': 4, 'ch_max_simul_firing': 8, 'wlen': 6, 'prominence': 10}
Recording test 总样本数: 6000000
Recording segment 总样本数: 6000000
时间范围: 600 - 1200 秒
采样率: 10000.0 Hz

  traces 形状 (n_samples, n_channels): (1000, 49)

测试窗口 1:
  时间范围: 1185.55 - 1185.65 秒
  start_frame: 5855529, end_frame: 5856529
  窗口帧数: 1000
  traces 形状 (n_samples, n_channels): (1000, 49)
  时间点数: 1000, 通道数: 49
  检测到的 spikes: 233
  Spikes 矩阵形状: (1000, 49), 1 的数量: 233.0
  前5个 spike 位置: [[ 1 43]
 [ 9 16]
 [17 21]
 [20 17]
 [21 21]]
  Spikes 矩阵统计: min=0.0, max=1.0, sum=233.0
  traces 形状 (n_samples, n_channels): (1000, 49)

测试窗口 2:
  时间范围: 776.05 - 776.15 秒
  start_frame: 1760504, end_frame: 1761504
  窗口帧数: 1000
  traces 形状 (n_samples, n_channels): (1000, 49)
  时间点数: 1000, 通道数: 49
  检测到的 spikes: 251
  Spikes 矩阵形状: (1000, 49), 1 的数量: 251.0
  前5个 spike 位置: [[ 3  2]
 [ 5 14]
 [ 6 40]
 [ 7 28]
 [ 7 43]]
  Spikes 矩阵统计: min=0.0, max=1.0, sum=251.0
  traces 形状 (n_sampl

## 7. 执行时间计算


In [23]:
print("=" * 80)
print("执行时间计算")
print("=" * 80)

# 初始化结果
timing_results = []

# 使用可用的 cliques（限制为5个用于测试）
test_cliques = [cliques[cid] for cid in available_cliques[:min(5, len(available_cliques))]]
test_clique_ids = available_cliques[:min(5, len(available_cliques))]

print(f"Testing with {len(test_clique_ids)} cliques: {test_clique_ids}")

# 获取 recording_segment 的总样本数（用于索引边界检查）
recording_segment_num_samples = recording_segment.get_num_samples()
print(f"Recording segment 总样本数: {recording_segment_num_samples}")
print(f"时间范围: {start_time} - {end_time} 秒")
print(f"采样率: {sampling_rate} Hz\n")

# 处理每个窗口大小
for window_size_ms in window_sizes_ms:
    print(f"\n测试处理窗口大小: {window_size_ms} ms")
    time_window_seconds = window_size_ms / 1000.0
    
    # 测试单个 clique 处理时间
    single_clique_times = []
    all_cliques_serial_times = []
    debug_info_list = []  # 用于调试：记录检测信息
    
    for run_idx in tqdm(range(n_runs_per_window), desc=f"Window {window_size_ms}ms", leave=False):
        # 在时间段内随机起始时间
        random_start_time = np.random.uniform(start_time, end_time - time_window_seconds)
        # 计算相对于 recording_segment 的帧索引
        start_frame_absolute = int(random_start_time * sampling_rate)
        start_frame = start_frame_absolute - start_sample  # 转换为相对于 recording_segment 的索引
        
        # 确保索引在有效范围内
        if start_frame < 0:
            start_frame = 0
        if start_frame >= recording_segment_num_samples:
            start_frame = recording_segment_num_samples - 1
        
        # 测试单个 clique（使用第一个 clique 作为示例）
        if len(test_clique_ids) > 0:
            clique_id = test_clique_ids[0]
            clique = test_cliques[0]
            clique_channels = list(set(clique.device_channel_indices))
            recording_clique = recording_segment.select_channels(channel_ids=clique_channels)
            
            try:
                result = process_single_clique_timing(
                    recording_clique=recording_clique,
                    autosort_model=clique_models[clique_id],
                    calibration_results=clique_calibration_results[clique_id],
                    start_frame=start_frame,  # 使用相对索引
                    time_window_seconds=time_window_seconds,
                    detection_params=detection_params,
                    window_params=window_params,
                    device=device,
                    return_n_spikes=True,
                )
                single_clique_time, debug_info = result
                debug_info_list.append(debug_info)
                # 记录所有时间（包括数据加载和检测的时间）
                single_clique_times.append(single_clique_time)
            except Exception as e:
                print(f"  Error in single clique timing: {e}")
                import traceback
                traceback.print_exc()
                # 不添加 NaN，只跳过
        
        # 测试所有 cliques 的串行处理
        serial_start = time.time()
        for clique_id, clique in zip(test_clique_ids, test_cliques):
            clique_channels = list(set(clique.device_channel_indices))
            recording_clique = recording_segment.select_channels(channel_ids=clique_channels)
            try:
                # 使用相同的相对索引 start_frame
                process_single_clique_timing(
                    recording_clique=recording_clique,
                    autosort_model=clique_models[clique_id],
                    calibration_results=clique_calibration_results[clique_id],
                    start_frame=start_frame,  # 已经是相对索引
                    time_window_seconds=time_window_seconds,
                    detection_params=detection_params,
                    window_params=window_params,
                    device=device,
                )
            except Exception as e:
                pass
        serial_end = time.time()
        all_cliques_serial_times.append(serial_end - serial_start)
    
    # 存储结果
    for run_idx in range(n_runs_per_window):
        timing_results.append({
            'window_size_ms': window_size_ms,
            'run': run_idx + 1,
            'single_clique_time': single_clique_times[run_idx] if run_idx < len(single_clique_times) else np.nan,
            'all_cliques_serial_time': all_cliques_serial_times[run_idx],
        })
    
    # 显示统计信息
    if len(single_clique_times) > 0 and len(debug_info_list) > 0:
        avg_spikes_detected = np.mean([info['n_spikes_detected'] for info in debug_info_list])
        avg_waveforms_extracted = np.mean([info['n_waveforms_extracted'] for info in debug_info_list])
        avg_spikes_after_noise = np.mean([info['n_spikes_after_noise_classifier'] for info in debug_info_list])
        avg_timepoints = np.mean([info['n_timepoints'] for info in debug_info_list])
        
        print(f"  单个 clique: {np.mean(single_clique_times):.6f} ± {np.std(single_clique_times):.6f} 秒")
        print(f"    阈值检测: 平均 {avg_spikes_detected:.1f} 个 spikes (时间点: {avg_timepoints:.0f})")
        print(f"    波形提取: 平均 {avg_waveforms_extracted:.1f} 个波形")
        print(f"    噪声分类后: 平均 {avg_spikes_after_noise:.1f} 个 spikes")
    else:
        print(f"  单个 clique: 无有效数据")
    print(f"  串行处理（所有 cliques）: {np.mean(all_cliques_serial_times):.6f} ± {np.std(all_cliques_serial_times):.6f} 秒")


执行时间计算
Testing with 5 cliques: [0, 1, 2, 3, 4]
Recording segment 总样本数: 6000000
时间范围: 600 - 1200 秒
采样率: 10000.0 Hz


测试处理窗口大小: 100 ms


  单个 clique: 0.027985 ± 0.023177 秒
    阈值检测: 平均 212.8 个 spikes (时间点: 1000)
    波形提取: 平均 205.9 个波形
    噪声分类后: 平均 20.2 个 spikes
  串行处理（所有 cliques）: 0.025487 ± 0.001630 秒

测试处理窗口大小: 200 ms


  单个 clique: 0.035263 ± 0.008323 秒
    阈值检测: 平均 413.7 个 spikes (时间点: 2000)
    波形提取: 平均 407.2 个波形
    噪声分类后: 平均 43.5 个 spikes
  串行处理（所有 cliques）: 0.036174 ± 0.001525 秒

测试处理窗口大小: 300 ms


  单个 clique: 0.046251 ± 0.010486 秒
    阈值检测: 平均 632.0 个 spikes (时间点: 3000)
    波形提取: 平均 625.1 个波形
    噪声分类后: 平均 67.5 个 spikes
  串行处理（所有 cliques）: 0.051259 ± 0.005181 秒

测试处理窗口大小: 400 ms


  单个 clique: 0.053574 ± 0.007642 秒
    阈值检测: 平均 847.5 个 spikes (时间点: 4000)
    波形提取: 平均 841.3 个波形
    噪声分类后: 平均 92.0 个 spikes
  串行处理（所有 cliques）: 0.061562 ± 0.001094 秒

测试处理窗口大小: 500 ms


  单个 clique: 0.069854 ± 0.010337 秒
    阈值检测: 平均 1055.1 个 spikes (时间点: 5000)
    波形提取: 平均 1050.3 个波形
    噪声分类后: 平均 115.5 个 spikes
  串行处理（所有 cliques）: 0.074140 ± 0.001606 秒

测试处理窗口大小: 1000 ms


  单个 clique: 0.105074 ± 0.013326 秒
    阈值检测: 平均 2129.6 个 spikes (时间点: 10000)
    波形提取: 平均 2124.9 个波形
    噪声分类后: 平均 228.5 个 spikes
  串行处理（所有 cliques）: 0.138569 ± 0.001819 秒


## 8. 保存和显示结果


In [24]:
# 转换为 DataFrame 并保存
timing_df = pd.DataFrame(timing_results)
timing_df_path = Path(base_save_dir) / "clique_processing_timing.csv"
timing_df.to_csv(timing_df_path, index=False)
print(f"\n时间结果已保存到: {timing_df_path}")

# 打印摘要
print(f"\n{'='*80}")
print(f"时间结果摘要:")
print(f"{'='*80}")
summary = timing_df.groupby('window_size_ms').agg({
    'single_clique_time': ['mean', 'std'],
    'all_cliques_serial_time': ['mean', 'std'],
}).round(4)
print(summary)

# 显示详细结果
print(f"\n{'='*80}")
print(f"详细结果:")
print(f"{'='*80}")
print(timing_df.head(10))



时间结果已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_processing_timing.csv

时间结果摘要:
               single_clique_time         all_cliques_serial_time        
                             mean     std                    mean     std
window_size_ms                                                           
100                        0.0280  0.0238                  0.0255  0.0017
200                        0.0353  0.0085                  0.0362  0.0016
300                        0.0463  0.0108                  0.0513  0.0053
400                        0.0536  0.0078                  0.0616  0.0011
500                        0.0699  0.0106                  0.0741  0.0016
1000                       0.1051  0.0137                  0.1386  0.0019

详细结果:
   window_size_ms  run  single_clique_time  all_cliques_serial_time
0             100    1            0.125133 

## 9. 可视化结果（可选）


In [26]:
# 创建可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 单个 clique 处理时间
ax1 = axes[0]
for window_size in window_sizes_ms:
    data = timing_df[timing_df['window_size_ms'] == window_size]['single_clique_time'].dropna()
    if len(data) > 0:
        ax1.scatter([window_size] * len(data), data, alpha=0.5, s=20)
        mean_time = data.mean()
        ax1.scatter(window_size, mean_time, color='red', s=100, marker='x', linewidths=2)

ax1.set_xlabel('Window Size (ms)')
ax1.set_ylabel('Processing Time (seconds)')
ax1.set_title('Single Clique Processing Time')
ax1.grid(True, alpha=0.3)

# 所有 cliques 串行处理时间
ax2 = axes[1]
for window_size in window_sizes_ms:
    data = timing_df[timing_df['window_size_ms'] == window_size]['all_cliques_serial_time']
    if len(data) > 0:
        ax2.scatter([window_size] * len(data), data, alpha=0.5, s=20)
        mean_time = data.mean()
        ax2.scatter(window_size, mean_time, color='red', s=100, marker='x', linewidths=2)

ax2.set_xlabel('Window Size (ms)')
ax2.set_ylabel('Processing Time (seconds)')
ax2.set_title('All Cliques Serial Processing Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()

# 保存图像
fig_path = Path(base_save_dir) / "clique_processing_timing_plot.pdf"
fig.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"\n可视化结果已保存到: {fig_path}")

plt.show()



可视化结果已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/02_simulation_data/02_Neuropixel_384_channels/data_generation/recording_neuropixels_150_Neuron_3600s/autosort_input/clique_processing_timing_plot.pdf


## 10. 清理内存


In [ ]:
# 清理内存
del clique_models, clique_calibration_results, recording_segment
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("内存清理完成")
